In [1]:
## Part 1: Load pre-processed data

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
# --- Path setup ---
project_root = Path("/home/jovyan/project-WaterDig")

# Pre-processed data files
data_dir = project_root / "processed_data" 

# Output directory
results_dir = project_root / "model_data"/ "mann_kendall"
results_dir.mkdir(exist_ok=True)

print("Reading pre-processed data:", data_dir)
print("Results will be saved to:", results_dir)

Reading pre-processed data: /home/jovyan/project-WaterDig/processed_data
Results will be saved to: /home/jovyan/project-WaterDig/model_data/mann_kendall


In [4]:
#--- Read pre-processed climate data ---
data_monthly = pd.read_csv(data_dir / "Climate_data_monthly.csv")
data_seasonal = pd.read_csv(data_dir / "Climate_data_seasonal.csv")
data_yearly  = pd.read_csv(data_dir / "Climate_data_yearly.csv")

In [5]:
## Part 2: Mann - Kendall trend test

In [6]:
!pip install pymannkendall

Defaulting to user installation because normal site-packages is not writeable


In [7]:
import pymannkendall as mk

In [8]:
#--- Monthly MK ---

In [9]:
data_monthly["date"] = pd.to_datetime(
    dict(year=data_monthly.year,
         month=data_monthly.month,
         day=1)
)

data_monthly = data_monthly.set_index("date").sort_index()
data_monthly.head()

,year,month,precipitation(mm),snow_depth(cm),mean_temperature(°C),min_temperature(°C),max_temperature(°C)
date,,,,,,,
1991-01-01,1991,1,68.9,43.000000,-3.632258,-6.829032,-0.506452
1991-02-01,1991,2,16.4,53.178571,-7.500000,-10.735714,-4.532143
1991-03-01,1991,3,31.2,45.903226,-0.980645,-3.558065,1.929032
1991-04-01,1991,4,14.1,6.400000,3.433333,-1.166667,8.286667
1991-05-01,1991,5,28.8,0.000000,7.241935,2.487097,12.177419


In [10]:
monthly_results = []
# Exclude non-variables
exclude_vars = ["year", "month"]

for var in data_monthly.columns:
    
    if var in exclude_vars:
        continue
    
    for m in range(1, 13):
        
        subset = data_monthly[data_monthly["month"] == m][var].dropna()
        
        if len(subset) > 10:  # avoid short time series
            
            res = mk.original_test(subset)
            
            monthly_results.append({
                "variable": var,
                "month": m,
                "trend": res.trend,
                "p_value": res.p,
                "sen_slope": res.slope,
                "z": res.z
            })

monthly_results_MK = pd.DataFrame(monthly_results)

monthly_results_MK

,variable,month,trend,p_value,sen_slope,z
0,precipitation(mm),1,no trend,0.394107,-0.353587,-0.852193
1,precipitation(mm),2,no trend,0.417406,0.282909,0.810929
2,precipitation(mm),3,no trend,0.475464,-0.175000,-0.713618
3,precipitation(mm),4,no trend,0.242747,0.362500,1.168148
4,precipitation(mm),5,no trend,0.649742,0.288942,0.454120
5,precipitation(mm),6,no trend,0.592551,-0.508882,-0.535143
6,precipitation(mm),7,no trend,0.592551,-0.437500,-0.535143
7,precipitation(mm),8,no trend,0.570323,-0.554167,-0.567576
8,precipitation(mm),9,no trend,0.896765,0.117361,0.129749
9,precipitation(mm),10,no trend,0.733446,-0.234630,-0.340546


In [11]:
monthly_results_MK.to_csv(
    results_dir / "monthly_trends.csv",
    index=False
)

In [12]:
#--- Seasonal MK ---

In [13]:
season_order = ["Spring", "Summer", "Autumn", "Winter"]

data_seasonal["season"] = (
    data_seasonal["season"]
    .str.strip()
    .str.capitalize()
)

In [14]:
seasonal_results = []

variables = [c for c in data_seasonal.columns if c not in ["year", "season"]]

for var in variables:
    for season in season_order:
        
        series = (
            data_seasonal
            .loc[data_seasonal["season"] == season]
            .set_index("year")[var]
            .dropna()
        )
        
        # MK needs enough data points
        if len(series) < 8:
            continue
        
        res = mk.original_test(series)
        
        seasonal_results.append({
            "variable": var,
            "season": season,
            "n_years": len(series),
            "trend": res.trend,
            "p_value": res.p,
            "sen_slope": res.slope,
            "z": res.z
        })

seasonal_trends_MK = pd.DataFrame(seasonal_results)
seasonal_trends_MK

,variable,season,n_years,trend,p_value,sen_slope,z
0,precipitation(mm),Spring,32,no trend,0.592551,0.288462,0.535143
1,precipitation(mm),Summer,32,no trend,0.306953,-1.488929,-1.021637
2,precipitation(mm),Autumn,32,no trend,0.733446,-0.524359,-0.340546
3,precipitation(mm),Winter,33,no trend,0.360627,0.781970,0.914170
4,snow_depth(cm),Spring,32,no trend,0.291851,-0.106806,-1.054069
5,snow_depth(cm),Summer,32,no trend,1.000000,0.000000,0.000000
6,snow_depth(cm),Autumn,32,decreasing,0.002678,-0.024882,-3.002413
7,snow_depth(cm),Winter,33,no trend,0.344578,-0.160128,-0.945159
8,mean_temperature(°C),Spring,32,no trend,0.094861,0.038890,1.670295
9,mean_temperature(°C),Summer,32,no trend,0.057785,0.041946,1.897325


In [15]:
seasonal_trends_MK.to_csv(
    results_dir / "seasonal_trends.csv",
    index=False
)

In [16]:
#--- Yearly MK ---

In [17]:
yearly_results = []

for var in data_yearly.columns:
    if var == "year":
        continue

    series = (
        data_yearly
        .set_index("year")[var]
        .dropna()
    )

    # Minimum length check (good practice)
    if len(series) < 8:
        continue

    res = mk.original_test(series)

    yearly_results.append({
        "variable": var,
        "trend": res.trend,
        "p_value": res.p,
        "sen_slope": res.slope,
        "z": res.z,
        "n_years": len(series)
    })

yearly_trends_MK = pd.DataFrame(yearly_results)
yearly_trends_MK

,variable,trend,p_value,sen_slope,z,n_years
0,precipitation(mm),no trend,0.816215,-0.533523,-0.232416,33
1,snow_depth(cm),no trend,0.344578,-0.066180,-0.945159,33
2,mean_temperature(°C),increasing,0.016322,0.037974,2.401634,33
3,min_temperature(°C),increasing,0.033776,0.034603,2.122735,33
4,max_temperature(°C),increasing,0.009666,0.039616,2.587567,33


In [18]:
yearly_trends_MK.to_csv(
    results_dir / "yearly_trends.csv",
    index=False
)